![Finance Toolkit](https://github.com/JerBouma/FinanceToolkit/assets/46355364/198d47bd-e1b3-492d-acc4-5d9f02d1d009)

**The FinanceToolkit** is an open-source toolkit in which all relevant financial ratios (100+), indicators and performance measurements are written down in the most simplistic way allowing for complete transparency of the calculation method. This allows you to not have to rely on metrics from other providers and, given a financial statement, allow for efficient manual calculations. This leads to one uniform method of calculation being applied that is available and understood by everyone.

# Installation
To install the FinanceToolkit it simply requires the following:

```
pip install financetoolkit -U
```

From within Python use:

```python
from financetoolkit import Toolkit
```
 
To be able to get started, you need to obtain an API Key from FinancialModelingPrep. This is used to gain access to 30+ years of financial statement both annually and quarterly. Note that the Free plan is limited to 250 requests each day, 5 years of data and only features companies listed on US exchanges.

___ 

<b><div align="center">Obtain an API Key from FinancialModelingPrep <a href="https://www.jeroenbouma.com/fmp" target="_blank">here</a>.</div></b>
___

Through the link you are able to subscribe for the free plan and also premium plans at a **15% discount**. This is an affiliate link and thus supports the project at the same time. I have chosen FinancialModelingPrep as a source as I find it to be the most transparent, reliable and at an affordable price. When you notice that data is inaccurate or have any other issue related to the data, note that I simply provide the means to access this data and I am not responsible for the accuracy of the data itself. For this, use <a href="https://site.financialmodelingprep.com/contact" target="_blank">their contact form</a> or provide the data yourself.

In [ ]:
import pandas as pd

from financetoolkit import Toolkit

API_KEY = "FINANCIAL_MODELING_PREP_API_KEY"

The Econometrics module provides a broad set of regression, hypothesis-testing, time-series and panel-data methods -- Ordinary Least Squares (OLS) and other regression estimators, unit root and cointegration tests, Granger causality, causal inference (IV/DiD/RDD/PSM/Synthetic Control), panel data (Fixed/Random Effects) and time-series forecasting (ARIMA, VAR, VECM). It is accessed through `toolkit.econometrics` and requires the optional `financetoolkit[econometrics]` extra (`pip install financetoolkit[econometrics]`), which pulls in `statsmodels` and `linearmodels`.

Rather than looking at each method in isolation, this notebook follows one investigation from start to finish: **is Apple's stock actually tied to its chip suppliers and megacap peers, or is that just eyeballed pairwise correlation?** Along the way we bring in a deliberately wide set of tickers -- Apple's own RF/modem/foundry suppliers (`QCOM`, `SWKS`, `TSM`), other megacap tech (`MSFT`, `GOOGL`, `AMZN`, `META`, `NVDA`) and two names from unrelated sectors (`XOM`, `PG`) as a contrast -- and deliberately omit the Benchmark so nothing here is just "the whole market moving together".

In [ ]:
# A deliberately wide, varied universe: Apple's suppliers, megacap tech peers and
# two unrelated names (XOM, PG) for contrast. No Benchmark.
companies = Toolkit(
    ["AAPL", "TSM", "QCOM", "SWKS", "MSFT", "GOOGL", "AMZN", "META", "NVDA", "XOM", "PG"],
    api_key=API_KEY,
    start_date="2019-01-01",
    end_date="2023-01-01",
)

**Step 1 -- does anything here explain Apple's returns at all?** An Ordinary Least Squares (OLS) regression of Apple's weekly returns on every other ticker at once tells you which relationships survive once you control for all the others simultaneously -- not just a raw pairwise correlation, which can be misleading when the regressors are themselves correlated with each other. Beyond the coefficient, it reports the Standard Error, t-Statistic and P-Value for each regressor.

In [ ]:
companies.econometrics.get_ols(
    dependent_ticker="AAPL",
    independent_tickers=[
        "TSM", "QCOM", "SWKS", "MSFT", "GOOGL", "AMZN", "META", "NVDA", "XOM", "PG"
    ],
    period="weekly",
)

Only some of the initially plausible relationships survive: Apple's RF/modem chip suppliers `QCOM` and `SWKS` and megacap peers `MSFT` and `GOOGL` come out statistically significant (P-Value < 0.05), while `AMZN`, `META`, `NVDA` and the unrelated `XOM` do not. Two results stand out as counter-intuitive: `PG` (consumer staples, meant as an unrelated contrast) is significant, while Apple's own foundry partner `TSM` is not -- a first hint that multicollinearity between correlated regressors can hide or manufacture relationships in a multiple regression. `SWKS` has the strongest relationship of the group, so that is what we dig into next.

**Step 2 -- can we trust those P-Values?** OLS's t-tests assume (asymptotically) well-behaved residuals; wildly non-normal returns would undermine that. The **Jarque-Bera test** checks this directly for Apple and its three most relevant chip names.

In [ ]:
jarque_bera = companies.econometrics.get_jarque_bera_test(period="quarterly", within_period=False)

jarque_bera[["AAPL", "SWKS", "QCOM", "TSM"]]

None of the four are close to rejecting normality (all P-Values well above 0.05), so there is no red flag undermining the OLS significance calls above.

**Step 3 -- is the `SWKS` relationship predictive, or just same-period comovement?** `SWKS` had by far the strongest coefficient in the regression, but that only tells you the two moved together *in the same week*. **Granger causality** instead asks whether one ticker's *past* values help predict the other's returns beyond the other's own history -- evidence of a lead-lag relationship rather than true causation. `get_granger_causality` always computes every ordered pair in the Toolkit instance in one go, so we compute it once and select the `AAPL`/`SWKS` pair.

In [ ]:
granger_causality = companies.econometrics.get_granger_causality(period="weekly")

granger_causality.loc[[("AAPL", "SWKS"), ("SWKS", "AAPL")]]

Neither direction is significant (P-Values of 0.47 and 0.31). So the `SWKS`-`AAPL` relationship found in Step 1 is a *contemporaneous* one -- they tend to move in the same week -- not a predictive, lead-lag relationship in either direction.

**Step 4 -- what about the long run?** `TSM` is Apple's actual foundry partner, yet its *returns* were not significant in Step 1. Short-run returns and long-run price levels answer different questions, though -- two series can share a long-run equilibrium (**cointegration**) even without a short-run return relationship. Before testing that, the **Augmented Dickey-Fuller (ADF)** test confirms the price levels actually need this treatment in the first place: its null hypothesis is that a series is *non-stationary* (has a unit root), which price levels normally are, unlike returns.

In [ ]:
adf = companies.econometrics.get_augmented_dickey_fuller(period="quarterly")

display(adf[["AAPL", "TSM"]])

cointegration = companies.econometrics.get_engle_granger_cointegration(period="quarterly")

display(cointegration.loc[[("AAPL", "TSM"), ("TSM", "AAPL")]])

Apple's price fails to reject the unit root (P-Value 0.53), as expected for a price level. `TSM`'s price is a more borderline case here given the short quarterly sample (only 11-15 observations), rejecting at the 5% level -- worth treating with some caution rather than at face value. Either way, the **Engle-Granger** test finds no cointegrating relationship between the two (P-Value 0.99): even Apple's own foundry partner shows no detectable long-run price equilibrium with Apple over this window, at least not with this few observations.

**Step 5 -- zoom out: does the whole panel share a common factor?** Rather than one pair at a time, `get_fixed_effects` treats every remaining ticker as an entity in a genuine panel and asks whether they share a common sensitivity to `SWKS`, after removing each entity's own time-invariant average return.

In [ ]:
fixed_effects = companies.econometrics.get_fixed_effects(
    independent_tickers="SWKS",
    dependent_tickers=[
        "TSM", "QCOM", "MSFT", "GOOGL", "AMZN", "META", "NVDA", "XOM", "PG"
    ],
    period="weekly",
)

The whole panel -- including `XOM` and `PG`, the two names picked specifically for being unrelated -- shares a highly significant sensitivity to `SWKS` (P-Value 0.0). That is the tell: since the Benchmark was deliberately excluded from this analysis, `SWKS`'s return here is standing in for general market-wide comovement rather than a genuine semiconductor-supply-chain effect. A cleaner version of this test would include the market factor explicitly and check what, if anything, is left over -- a good next step to try with `include_benchmark=True` on `get_ols` above.

Using the individual models with your own DataFrames is also a possibility thanks to the architecture of the Finance Toolkit.

In [ ]:
import numpy as np

from financetoolkit.econometrics import diagnostics_model, regression_model

np.random.seed(42)

factor = pd.Series(np.random.normal(0, 1, 100), name="Factor")
asset_return = 2.5 * factor + pd.Series(np.random.normal(0, 0.5, 100))
asset_return.name = "Asset Return"

result = regression_model.get_ols(asset_return, factor)

display(regression_model.regression_summary_table(result))

display(diagnostics_model.get_jarque_bera_test(asset_return))